In [1]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import quote

keyword = "숏폼"
query = quote(keyword)

url = f"https://search.daum.net/search?w=news&q={query}"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
    "Accept-Language": "ko-KR,ko;q=0.9"
}

res = requests.get(url, headers=headers, timeout=10)
print(res.status_code)
print(res.text[:500])

200
<!doctype html>
<html xmlns="http://www.w3.org/1999/xhtml" lang="ko">
<head profile="http://a9.com/-/spec/opensearch/1.1/">
<meta http-equiv="content-Type" content="text/html;charset=utf-8" />
<meta http-equiv="X-UA-Compatible" content="IE=edge" />
<meta name="autocomplete" content="off" />
<meta name="referrer" content="always">
<meta name="format-detection" content="telephone=no,address=no,email=no" />
<meta property="og:title" content="숏폼 &ndash; Daum 검색" />
<meta property="og:url" content="h


In [3]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
from urllib.parse import quote
import pandas as pd
import time

keyword = "숏폼"
query = quote(keyword)

articles = []

options = Options()
options.add_argument("--headless=new")
options.add_argument("--disable-gpu")
options.add_argument("--no-sandbox")
options.add_argument("--window-size=1920,1080")
options.add_argument("--lang=ko-KR")
options.add_argument(
    "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120 Safari/537.36"
)

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

for page in range(1, 51):
    url = f"https://search.daum.net/search?w=news&q={query}&p={page}"
    print(f"{page}페이지 수집 중")

    driver.get(url)
    time.sleep(2)

    soup = BeautifulSoup(driver.page_source, "html.parser")

    links = soup.select("a[href*='v.daum.net'], a[href*='news.v.daum.net']")

    if len(links) == 0:
        print("기사 링크 없음, 수집 종료")
        break

    for tag in links:
        title = tag.get_text(strip=True)
        link = tag.get("href")

        if not title:
            continue

        if keyword not in title:
            continue

        articles.append({
            "keyword": keyword,
            "title": title,
            "link": link
        })

    time.sleep(1)

driver.quit()

df = pd.DataFrame(articles)

df = df.drop_duplicates(subset=["title", "link"])
df.to_csv("daum_news_title_filtered.csv", index=False, encoding="utf-8-sig")

print("수집 완료")
print("수집 건수:", len(df))

df.head()

1페이지 수집 중
2페이지 수집 중
3페이지 수집 중
4페이지 수집 중
5페이지 수집 중
6페이지 수집 중
7페이지 수집 중
8페이지 수집 중
9페이지 수집 중
10페이지 수집 중
11페이지 수집 중
12페이지 수집 중
13페이지 수집 중
14페이지 수집 중
15페이지 수집 중
16페이지 수집 중
17페이지 수집 중
18페이지 수집 중
19페이지 수집 중
20페이지 수집 중
21페이지 수집 중
22페이지 수집 중
23페이지 수집 중
24페이지 수집 중
25페이지 수집 중
26페이지 수집 중
27페이지 수집 중
28페이지 수집 중
29페이지 수집 중
30페이지 수집 중
31페이지 수집 중
32페이지 수집 중
33페이지 수집 중
34페이지 수집 중
35페이지 수집 중
36페이지 수집 중
37페이지 수집 중
38페이지 수집 중
39페이지 수집 중
40페이지 수집 중
41페이지 수집 중
42페이지 수집 중
43페이지 수집 중
44페이지 수집 중
45페이지 수집 중
46페이지 수집 중
47페이지 수집 중
48페이지 수집 중
49페이지 수집 중
50페이지 수집 중
수집 완료
수집 건수: 857


,keyword,title,link
0,숏폼,"비피엠지, 요리 게임 '마이리틀셰프'숏폼드라마 공개",http://v.daum.net/v/20260514103753411
1,숏폼,(지디넷코리아=정진성 기자)비피엠지가 글로벌 요리 시뮬레이션 게임 '마이리틀셰프' ...,http://v.daum.net/v/20260514103753411
2,숏폼,"비피엠지, 요리 게임 '마이리틀셰프'숏폼드라마 공개…IP 확장",http://v.daum.net/v/20260514173900976
3,숏폼,"비피엠지, ‘마이리틀셰프’숏폼드라마 공개…게임 IP 확장 본격화",http://v.daum.net/v/20260514134850763
4,숏폼,"비피엠지, 요리 게임 '마이리틀셰프'숏폼드라마 공개",http://v.daum.net/v/20260514095315573


In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
from urllib.parse import quote
import pandas as pd
import time

keyword = "숏폼"

options = Options()
options.add_argument("--headless=new")
options.add_argument("--disable-gpu")
options.add_argument("--no-sandbox")
options.add_argument("--window-size=1920,1080")

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

dates = pd.date_range("2021-05-01", "2026-04-30", freq="MS")

articles = []

for d in dates:
    start = d.strftime("%Y.%m.%d")
    end = (d + pd.offsets.MonthEnd(0)).strftime("%Y.%m.%d")

    print(f"\n수집 기간: {start} ~ {end}")

    for page in range(1, 101):
        url = f"https://search.daum.net/search?w=news&q={quote(keyword)}&period=u&sd={start}&ed={end}&p={page}"
        driver.get(url)
        time.sleep(1.5)

        soup = BeautifulSoup(driver.page_source, "html.parser")

        links = soup.select("a[href*='v.daum.net'], a[href*='news.v.daum.net']")

        if len(links) == 0:
            break

        for tag in links:
            title = tag.get_text(strip=True)
            link = tag.get("href")

            if not title:
                continue

            if keyword not in title:
                continue

            try:
                driver.get(link)
                time.sleep(1)

                article_soup = BeautifulSoup(driver.page_source, "html.parser")

                date_tag = article_soup.select_one("meta[property='article:published_time']")
                pub_date = date_tag.get("content") if date_tag else ""

                articles.append({
                    "keyword": keyword,
                    "title": title,
                    "pub_date": pub_date,
                    "link": link
                })

                print("수집:", title[:40])

            except Exception as e:
                print("기사 날짜 수집 실패:", e)
                continue

df = pd.DataFrame(articles)

df = df.drop_duplicates(subset=["title", "link"])

df["pub_date"] = pd.to_datetime(df["pub_date"], errors="coerce")
df["year"] = df["pub_date"].dt.year
df["month"] = df["pub_date"].dt.to_period("M").astype(str)

df.to_csv("daum_news_title_date_5years.csv", index=False, encoding="utf-8-sig")

driver.quit()

print("수집 완료")
print("총 기사 수:", len(df))
df.head()


수집 기간: 2021.05.01 ~ 2021.05.31
수집: 비피엠지, 요리 게임 '마이리틀셰프'숏폼드라마 공개
수집: (지디넷코리아=정진성 기자)비피엠지가 글로벌 요리 시뮬레이션 게임 '마이
수집: 비피엠지, 요리 게임 '마이리틀셰프'숏폼드라마 공개…IP 확장
수집: 비피엠지, ‘마이리틀셰프’숏폼드라마 공개…게임 IP 확장 본격화
수집: 비피엠지, 요리 게임 '마이리틀셰프'숏폼드라마 공개
수집: 버즈니 비스킷AI, 링크 하나로숏폼자동 제작
수집: /버즈니 | 서울=한스경제 전시현 기자 | 커머스 인공지능(AI) 기업 
수집: 버즈니, '비스킷AI' 기능 확대…뉴스·블로그 URL로숏폼만든다
수집: 버즈니는 인공지능(AI)숏폼자동 생성 서비스 'VISKIT AI(비스킷A
수집: 최서은,숏폼드라마 ‘아듀 솔로’ 혜수 역 출연
수집: 배우 최서은 링컴즈 제공 배우 최서은이숏폼드라마 ‘아듀 솔로’에 출연해 
